| título | projeto | versão | data | autores | status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| CRISP-DM — Fase 4: Modeling | Projeção da Taxa de Congestionamento — Justiça Estadual (GO) | 1.0 | 14-12-2025 | Júlio César e Lays de Freitas | Rascunho |


Esse Notebook contém a *Modelagem Preditiva (Baseline)*.

### BIBLIOTECAS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from dateutil.relativedelta import relativedelta
import warnings
warnings.filterwarnings('ignore')


### IMPORTAÇÃO E CONFIGURAÇÃO

In [2]:
# Carregamento dos dados processados na fase anterior
# Nota: Concatenamos treino e teste para refazer a divisão de forma TEMPORAL (passado vs futuro)

# Configurações visuais
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

try:
    df_train = pd.read_csv('datasets/train-processed.csv')
    df_test_split = pd.read_csv('datasets/test_split.csv')
    df_full = pd.concat([df_train, df_test_split], ignore_index=True)
except FileNotFoundError:
    # Caso não encontre os arquivos, usamos um dataframe simulado baseado na estrutura conhecida
    print("Arquivos não encontrados. Certifique-se de que a Fase 3 foi executada.")

# Garantir que a coluna de data é datetime
df_full['mes_ref'] = pd.to_datetime(df_full['mes_ref'])
#df_full['Taxa de Congestionamento_mes'] = df_full['Taxa de Congestionamento_mes (%)'] * 100  # Converter para porcentagem

# Ordenar por unidade e data
df_full = df_full.sort_values(by=['comarca', 'serventia', 'mes_ref'])

print(f"Total de registros carregados: {len(df_full)}")
print(f"Período dos dados: de {df_full['mes_ref'].min().date()} até {df_full['mes_ref'].max().date()}")

Total de registros carregados: 571917
Período dos dados: de 2022-01-01 até 2025-10-01


In [3]:
df_full.sample(5)

,comarca,serventia,Distribuídos_mes,Baixados_mes,Pendentes_mes,Taxa de Congestionamento_mes (%),mes_ref
458734,TRIBUNAL DE JUSTIÇA,GABINETE DES. GERSON SANTANA CINTRA,107,0,1,100.00,2025-06-01
356145,ACREÚNA,"1ª Vara Judicial (Família e Sucessões, Infânci...",61,0,5,100.00,2025-09-01
355297,GOIÂNIA,6ª Vara de Família,121,2,6,75.00,2024-06-01
4540,PLANALTINA,"1ª Vara (Cível, de Família, Sucessões e da Inf...",151,3,29,90.62,2024-09-01
97765,MONTES CLAROS DE GOIÁS,Vara Judicial,45,0,5,100.00,2024-01-01


### REFINAMENTO DOS DADOS

In [4]:
# Vamos aplicar as regras de negócio para limpar a base antes do split temporal

# 1. Filtro de Tipo de Unidade: Remover CEJUSCs
# CEJUSCs costumam ter taxas de 0% ou 100% devido a mutirões
filtro_cejusc = ~df_full['serventia'].str.contains('CEJUSC|CENTRO JUDICIÁRIO', case=False, na=False)

# 2. Filtro de Volume (Tratamento de Zeros/Pequenas Amostras)
# Regra: Unidades com menos de 10 processos movimentados (Pendentes + Baixados) no mês são instáveis
# Se Pendentes=1 e Baixados=0 -> Taxa 100%. Se mês seguinte Pendentes=0 e Baixados=1 -> Taxa 0%.
df_full['volume_total'] = df_full['Pendentes_mes'] + df_full['Baixados_mes']
filtro_volume = df_full['volume_total'] >= 10

# Aplicação dos Filtros
df_refinado = df_full[filtro_cejusc & filtro_volume].copy()

# Estatísticas do Refinamento
total_original = len(df_full)
total_refinado = len(df_refinado)
removidos = total_original - total_refinado

print(f"Registros Originais: {total_original}")
print(f"Registros Após Refinamento: {total_refinado}")
print(f"Registros Removidos (Ruído): {removidos} ({removidos/total_original:.1%} da base)")

Registros Originais: 571917
Registros Após Refinamento: 397662
Registros Removidos (Ruído): 174255 (30.5% da base)


### FEATURE ENGINEERING E DIVISÃO TEMPORAL

In [5]:
# Para Regressão Linear funcionar com datas, precisamos converter a data para um número (ordinal ou timestamp)
# 1. Criação da Feature Numérica de Tempo
df_refinado['mes_ordinal'] = df_refinado['mes_ref'].apply(lambda x: x.toordinal())

# 2. Definição do Ponto de Corte (Split Temporal)
# Vamos usar os últimos 3 meses disponíveis como validação (Teste) e o resto como Treino
data_maxima = df_refinado['mes_ref'].max()
data_corte = data_maxima - relativedelta(months=3)

print(f"Data de Corte para Validação: {data_corte.date()}")

# Separação
df_treino = df_refinado[df_refinado['mes_ref'] <= data_corte].copy()
df_validacao = df_refinado[df_refinado['mes_ref'] > data_corte].copy()

print(f"Registros de Treino (Histórico): {len(df_treino)}")
print(f"Registros de Validação (Recente): {len(df_validacao)}")

Data de Corte para Validação: 2025-07-01
Registros de Treino (Histórico): 347128
Registros de Validação (Recente): 50534


### TREINAMENTO DO MODELO (LOOP POR UNIDADE)

In [6]:
# Iremos iterar por cada serventia, treinar uma regressão linear individual e prever
resultados_validacao = []
modelos_dict = {} # Dicionário para guardar os modelos treinados para uso futuro

# Identificar todas as combinações únicas de Comarca e Serventia
unidades = df_refinado[['comarca', 'serventia']].drop_duplicates()

print(f"Iniciando treinamento para {len(unidades)} unidades jurisdicionais...")

for index, row in unidades.iterrows():
    comarca_atual = row['comarca']
    serventia_atual = row['serventia']
    
    # Filtrar dados da unidade específica
    mask_treino = (df_treino['comarca'] == comarca_atual) & (df_treino['serventia'] == serventia_atual)
    mask_valid = (df_validacao['comarca'] == comarca_atual) & (df_validacao['serventia'] == serventia_atual)
    
    dados_treino = df_treino[mask_treino]
    dados_valid = df_validacao[mask_valid]
    
    # Regra de negócio: Precisamos de pelo menos 2 pontos para traçar uma reta
    if len(dados_treino) >= 2:
        # Preparar X e y
        X_train = dados_treino[['mes_ordinal']]
        y_train = dados_treino['Taxa de Congestionamento_mes (%)']
        
        # Instanciar e Treinar
        modelo = LinearRegression()
        modelo.fit(X_train, y_train)
        
        # Guardar modelo
        modelos_dict[(comarca_atual, serventia_atual)] = modelo
        
        # Se houver dados de validação, fazer a previsão para avaliar performance
        if len(dados_valid) > 0:
            X_valid = dados_valid[['mes_ordinal']]
            y_real = dados_valid['Taxa de Congestionamento_mes (%)']
            
            # Previsão
            y_pred = modelo.predict(X_valid)
            
            # Armazenar resultados para avaliação
            temp_df = dados_valid.copy()
            temp_df['Taxa_Prevista'] = y_pred
            # Clipar previsões entre 0 e 100 (pois é taxa %)
            temp_df['Taxa_Prevista'] = temp_df['Taxa_Prevista'].clip(0, 100)
            
            resultados_validacao.append(temp_df)

# Consolidar resultados de validação
df_avaliado = pd.concat(resultados_validacao, ignore_index=True)
print("Treinamento e Validação concluídos.")

Iniciando treinamento para 513 unidades jurisdicionais...
Treinamento e Validação concluídos.


In [7]:
#df_refinado.sample(10000).to_csv('datasets/full-refined-sample.csv', index=False)

### GERAÇÃO DE PREVISÕES FUTURAS

In [8]:
### GERAÇÃO DE PREVISÕES FUTURAS (3, 6 e 12 meses)

# Agora vamos projetar múltiplos horizontes temporais
horizontes = [3, 6, 12]  # meses
previsoes_futuras_por_horizonte = {horizon: [] for horizon in horizontes}

for (comarca, serventia), modelo in modelos_dict.items():
    for horizon in horizontes:
        # Gerar as datas futuras para cada horizonte
        datas_futuras = [data_maxima + relativedelta(months=i) for i in range(1, horizon + 1)]
        ordinais_futuros = np.array([d.toordinal() for d in datas_futuras]).reshape(-1, 1)
        
        # Prever para todas as datas do horizonte
        preds = modelo.predict(ordinais_futuros)
        
        for data, pred in zip(datas_futuras, preds):
            # Regra de negócio: Taxa não pode ser menor que 0 nem maior que 100
            pred_ajustado = max(0, min(100, pred))
            
            previsoes_futuras_por_horizonte[horizon].append({
                'comarca': comarca,
                'serventia': serventia,
                'data_futura': data,
                'horizonte_meses': horizon,
                'taxa_prevista': round(pred_ajustado, 2),
                'mes_no_horizonte': (data - data_maxima).days // 30  # Aproximação de meses
            })

# Consolidar resultados
dfs_futuros = {}
for horizon, previsoes in previsoes_futuras_por_horizonte.items():
    dfs_futuros[horizon] = pd.DataFrame(previsoes)

print("Amostra das previsões para diferentes horizontes:")
for horizon in [3, 6, 12]:
    print(f"\n=== Horizonte: {horizon} meses ===")
    display(dfs_futuros[horizon].head(3))

c:\Users\Administrador\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Administrador\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Administrador\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Administrador\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\Administrador\AppData\Local\Programs\Python\Python312\Lib\s

Amostra das previsões para diferentes horizontes:

=== Horizonte: 3 meses ===


,comarca,serventia,data_futura,horizonte_meses,taxa_prevista,mes_no_horizonte
0,ABADIÂNIA,Vara Judicial,2025-11-01,3,92.85,1
1,ABADIÂNIA,Vara Judicial,2025-12-01,3,93.21,2
2,ABADIÂNIA,Vara Judicial,2026-01-01,3,93.58,3



=== Horizonte: 6 meses ===


,comarca,serventia,data_futura,horizonte_meses,taxa_prevista,mes_no_horizonte
0,ABADIÂNIA,Vara Judicial,2025-11-01,6,92.85,1
1,ABADIÂNIA,Vara Judicial,2025-12-01,6,93.21,2
2,ABADIÂNIA,Vara Judicial,2026-01-01,6,93.58,3



=== Horizonte: 12 meses ===


,comarca,serventia,data_futura,horizonte_meses,taxa_prevista,mes_no_horizonte
0,ABADIÂNIA,Vara Judicial,2025-11-01,12,92.85,1
1,ABADIÂNIA,Vara Judicial,2025-12-01,12,93.21,2
2,ABADIÂNIA,Vara Judicial,2026-01-01,12,93.58,3


### AVALIAÇÃO DE MÉTRICAS (EVALUATION)

In [ ]:
# Vamos calcular o erro do nosso modelo nos dados de teste (validação)

mae = mean_absolute_error(df_avaliado['Taxa de Congestionamento_mes (%)'], df_avaliado['Taxa_Prevista'])
rmse = np.sqrt(mean_squared_error(df_avaliado['Taxa de Congestionamento_mes (%)'], df_avaliado['Taxa_Prevista']))

print("=== Performance Global do Modelo (Baseline: Regressão Linear) ===")
print(f"Erro Absoluto Médio (MAE): {mae:.2f} p.p.")
print(f"Raiz do Erro Quadrático Médio (RMSE): {rmse:.2f} p.p.")

# Análise de Erro: Onde o modelo errou mais?
df_avaliado['Erro_Absoluto'] = abs(df_avaliado['Taxa de Congestionamento_mes (%)'] - df_avaliado['Taxa_Prevista'])
top_erros = df_avaliado.sort_values(by='Erro_Absoluto', ascending=False).head(5)

print("\n=== Top 5 Maiores Erros de Previsão ===")
display(top_erros[['comarca', 'serventia', 'mes_ref', 'Taxa de Congestionamento_mes (%)', 'Taxa_Prevista', 'Erro_Absoluto']])

=== Performance Global do Modelo (Baseline: Regressão Linear) ===
Erro Absoluto Médio (MAE): 6.44 p.p.
Raiz do Erro Quadrático Médio (RMSE): 10.99 p.p.

=== Top 5 Maiores Erros de Previsão ===


,comarca,serventia,mes_ref,Taxa de Congestionamento_mes (%),Taxa_Prevista,Erro_Absoluto
45668,TRIBUNAL DE JUSTIÇA,GABINETE DES. ALEXANDRE BIZZOTTO,2025-10-01,100.0,3.895478,96.104522
45669,TRIBUNAL DE JUSTIÇA,GABINETE DES. ALEXANDRE BIZZOTTO,2025-10-01,100.0,3.895478,96.104522
46528,TRIBUNAL DE JUSTIÇA,GABINETE DES. WILSON DA SILVA DIAS,2025-10-01,100.0,6.231961,93.768039
46527,TRIBUNAL DE JUSTIÇA,GABINETE DES. WILSON DA SILVA DIAS,2025-10-01,100.0,6.231961,93.768039
46526,TRIBUNAL DE JUSTIÇA,GABINETE DES. WILSON DA SILVA DIAS,2025-10-01,100.0,6.231961,93.768039
